# Notebook 03 — Binary Cross-Entropy and Focal Loss

**Session:** Construct loss functions with different targets  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Implement Binary Cross-Entropy (BCE) and Focal Loss from scratch, inspect the modulating
factors $(1 - \hat{y})^\gamma$ and $\hat{y}^\gamma$, visualize how $\gamma \in \{0, 1, 2, 3, 4\}$
reshapes the loss curves, and quantify how Focal Loss prevents abundant easy examples
from drowning out rare hard misclassifications.

In [ ]:
import math

import matplotlib.pyplot as plt


def binary_crossentropy(y: int, y_hat: float, eps: float = 1e-12) -> float:
    """Compute numerically safe Binary Cross-Entropy loss for a single sample."""
    p = min(max(y_hat, eps), 1.0 - eps)
    return -y * math.log(p) - (1 - y) * math.log(1.0 - p)


predictions = [0.3, 0.8, 0.7, 0.4, 0.6, 0.8, 0.9, 0.2]
labels = [0, 0, 0, 0, 1, 1, 1, 1]
bce_losses = [binary_crossentropy(y, y_hat) for y, y_hat in zip(labels, predictions, strict=True)]
print("Balanced batch BCE losses:", [round(v, 4) for v in bce_losses])

## 1. Visualizing Modulating Factors and Focal Loss Curves

In [ ]:
def focal_loss_pos(y_hat: float, gamma: float) -> float:
    """Focal loss for positive class (y = 1)."""
    return -math.log(y_hat) * ((1.0 - y_hat) ** gamma)


def focal_loss_neg(y_hat: float, gamma: float) -> float:
    """Focal loss for negative class (y = 0)."""
    return -math.log(1.0 - y_hat) * (y_hat**gamma)


probs = [x / 100.0 for x in range(1, 100)]
gammas = [0, 1, 2, 3, 4]
colors = ["r", "g", "b", "c", "k"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for gamma, color in zip(gammas, colors, strict=True):
    axes[0].plot(probs, [focal_loss_pos(p, gamma) for p in probs], color, label=f"gamma={gamma}")
    axes[1].plot(probs, [focal_loss_neg(p, gamma) for p in probs], color, label=f"gamma={gamma}")

axes[0].set_title("Focal Loss (y = 1): -(1 - y_hat)^gamma * log(y_hat)")
axes[0].set_xlabel("Predicted probability (y_hat)")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].set_title("Focal Loss (y = 0): -y_hat^gamma * log(1 - y_hat)")
axes[1].set_xlabel("Predicted probability (y_hat)")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Class Imbalance Experiment: Easy Majority vs. Hard Minority

In [ ]:
def focal_loss(y: int, y_hat: float, gamma: float, eps: float = 1e-12) -> float:
    """Compute Focal Loss for binary classification."""
    p = min(max(y_hat, eps), 1.0 - eps)
    part1 = -y * ((1.0 - p) ** gamma) * math.log(p)
    part2 = -(1 - y) * (p**gamma) * math.log(1.0 - p)
    return part1 + part2


# 1 hard negative (y=0, y_hat=0.7) vs. 9 confident positives (y=1, y_hat in [0.7, 0.9])
imb_predictions = [0.7, 0.8, 0.7, 0.8, 0.8, 0.8, 0.9, 0.9, 0.8, 0.7]
imb_labels = [0, 1, 1, 1, 1, 1, 1, 1, 1, 1]

for gamma in range(5):
    losses = [
        focal_loss(y, y_hat, gamma) for y, y_hat in zip(imb_labels, imb_predictions, strict=True)
    ]
    hard_loss = losses[0]
    easy_sum = sum(losses[1:])
    print(f"gamma={gamma}: hard(y=0)={hard_loss:.3f}, sum_easy(y=1)={easy_sum:.3f}")